# FitCheck — FASHN 사이즈별 합성 확인

런타임 → 런타임 유형 변경 → GPU(T4)를 선택하고 셀을 순서대로 실행합니다.
기존 전신 착용 사진·기준 옷 총장/어깨너비/가슴단면·새 옷 S/M/L 치수·상품 사진을 준비합니다. 개인 00_ME 파일은 필요 없습니다.

웹 화면에서 치수 연결 → 사진의 5점과 측정 기준 확인 → 목표 윤곽 → FASHN 기본 합성 생성 → 초록 옷 영역 확인 순서로 진행합니다.
옷 영역 확인을 체크하면 S/M/L 후처리 결과가 나란히 표시됩니다. 사이즈 변경은 같은 기본 합성을 재사용하므로 GPU 합성을 다시 기다릴 필요 없습니다.
치수 적용 전 원본은 접힌 영역에 있습니다. 실제 비교는 판매 사이즈별 갤러리와 현재 선택 사이즈 결과에서 합니다.

길이·몸통 폭의 2D 변형 실험입니다. 어깨 변형, 소재·굴곡·실제 착용 핏은 검증 대상입니다. FASHN 엔진 자체에 cm 치수를 입력하는 방식은 아닙니다.
PC 변경사항을 GitHub에 Commit·Push한 뒤 사용하세요. 가중치는 기존 /content/fashn-weights 캐시를 재사용합니다.

In [ ]:
# 1. GPU 확인
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
# 2. 현재 코드와 FASHN 준비
import os
import sys
from pathlib import Path
APP = '/content/fitcheck-synthesis'
FASHN = '/content/fashn-vton-1.5'
WEIGHTS = '/content/fashn-weights'
REPOSITORY = 'https://github.com/eunzae02-glitch/virtual-fitting-app-master.git'
if not Path(APP).exists():
    subprocess.run(['git', 'clone', '--branch', 'main', REPOSITORY, APP], check=True)
else:
    origin = subprocess.check_output(['git', '-C', APP, 'remote', 'get-url', 'origin'], text=True).strip()
    if origin.rstrip('/').removesuffix('.git') != REPOSITORY.removesuffix('.git'):
        raise RuntimeError('APP에 다른 저장소가 있습니다. APP를 빈 새 폴더 경로로 바꿔주세요.')
    subprocess.run(['git', '-C', APP, 'pull', '--ff-only'], check=True)
if not Path(FASHN).exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/fashn-AI/fashn-vton-1.5.git', FASHN], check=True)
code = Path(APP, 'app', 'size_synthesis.py').read_text(encoding='utf-8')
if 'def adjust_sizes(' not in code:
    raise RuntimeError('사이즈 자동 적용 수정본이 없습니다. PC에서 Commit·Push한 뒤 이 셀을 다시 실행해주세요.')
subprocess.run(['git', '-C', APP, 'log', '-1', '--oneline'], check=True)

In [ ]:
# 3. FASHN·웹 화면·CPU 변형 라이브러리 설치
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', FASHN,
                'gradio==6.30.0', 'Pillow>=10,<13', 'numpy>=2,<3',
                'opencv-python>=4.10,<5', 'transformers>=4.40,<5'], check=True)
print('설치 완료')

In [ ]:
# 4. 기존 FASHN 합성·포즈·파서 가중치 준비. 기존 다운로드는 캐시 재사용.
subprocess.run([sys.executable, f'{FASHN}/scripts/download_weights.py', '--weights-dir', WEIGHTS],
               env=dict(os.environ, HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0'), check=True)
print('가중치 준비 완료')

In [ ]:
# 5. FASHN 사이즈 화면 실행 — gradio.live 링크를 여세요.
# 중지하면 이 셀이 만든 서버와 합성 자식 프로세스만 종료합니다.
import signal
env = dict(os.environ, FASHN_REPO=FASHN, FASHN_WEIGHTS=WEIGHTS, PYTHONUNBUFFERED='1')
fitcheck_size_server = subprocess.Popen(
    [sys.executable, '-u', f'{APP}/scripts/debug_size_preview.py', '--share'],
    cwd=APP, env=env, start_new_session=True)
try:
    code = fitcheck_size_server.wait()
    if code:
        raise RuntimeError(f'화면 서버 종료 코드: {code}. 위 오류를 확인해주세요.')
except KeyboardInterrupt:
    print('FASHN 사이즈 화면을 종료합니다.')
finally:
    if fitcheck_size_server.poll() is None:
        os.killpg(fitcheck_size_server.pid, signal.SIGTERM)
        try:
            fitcheck_size_server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(fitcheck_size_server.pid, signal.SIGKILL)
            fitcheck_size_server.wait()